# 05 - Ensemble, class weights and stacking for macro-F1

| Experiment | What is tuned | Reference |
|---|---|---|
| `ensemble` | weights of the averaged model probabilities | best single model |
| `scaling` | per-class multipliers on probabilities of the best single model | best single model |
| `ensemble+scaling` | both | best single model |
| `stacking` / `stacking_balanced` | LogReg meta-model on model probabilities + one-hot `domain` + typography artifacts (from `06`) | best single model |

- Inputs: `val_proba_*.csv` / `test_proba_*.csv` of the final `02` and `03` (`04` is not needed)
- Honest scores: parameters are chosen on one half of the validation split and scored on the other half (2-fold, both ways)
- Final parameters for test are chosen on the whole validation split

## Setup

In [1]:
import subprocess
import sys

REPO = "https://github.com/mohinska/ai_hw3_AIGenTextDetection.git"
REPO_DIR = "/tmp/repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [2]:
import itertools
import json
import os
import re

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import common
from common import LABELS, LABEL_COL, SEED, TEXT_COL
from features import artifact_features

## Config

In [3]:
MODELS = None                         # None = every model with both val and test probabilities
WEIGHT_STEP = 0.1                     # ensemble weight grid step (weights sum to 1)
SCALE_GRID = np.round(np.arange(0.5, 2.01, 0.1), 2)   # multipliers for p_human and p_fusion (p_ai = 1)
JOINT_WEIGHT_STEP, JOINT_SCALE_GRID = 0.25, np.round(np.arange(0.5, 2.01, 0.25), 2)  # coarser joint grid
STACK_C = 1.0                         # regularisation of the stacking meta-model
STACKERS = {"stacking": {"class_weight": None}, "stacking_balanced": {"class_weight": "balanced"}}
SUBMISSION_PATH = os.path.join(common.WORK_DIR, "submission_ensemble.csv")

## Load probabilities

In [4]:
train_df, test_df, sample_sub = common.load_data()
_, val = common.split_train_val(train_df)
y_val = val[LABEL_COL].to_numpy()

val_files = {re.sub(r"^val_proba_|\.csv$", "", os.path.basename(p)): p for p in common.find_outputs("val_proba_*.csv")}
test_files = {re.sub(r"^test_proba_|\.csv$", "", os.path.basename(p)): p for p in common.find_outputs("test_proba_*.csv")}
names = sorted(set(val_files) & set(test_files)) if MODELS is None else MODELS
assert names, "no probability files found: attach outputs of the final 02 and 03 as Kaggle input"

P_val = {n: common.load_proba(val_files[n], val["id"]) for n in names}
P_test = {n: common.load_proba(test_files[n], test_df["id"]) for n in names}


def macro(y, proba):
    return f1_score(y, np.array(LABELS)[proba.argmax(1)], average="macro")


single = pd.Series({n: macro(y_val, p) for n, p in P_val.items()}, name="val_macro_f1").sort_values(ascending=False)
REF = single.index[0]
single.round(4).to_frame()

,val_macro_f1
baseline_full,0.8951
encoder_full_chunks,0.8727


## Cross-fitted tuning

In [5]:
def simplex(n, step):
    """All weight vectors of length n with the given step that sum to 1."""
    k = round(1 / step)
    return [np.array(c) / k for c in itertools.product(range(k + 1), repeat=n) if sum(c) == k]


def scale_grid(values):
    """Per-class multipliers for p_human and p_fusion (p_ai stays 1)."""
    return [{"human": h, "fusion": f} for h in values for f in values]


def combine(P, params, idx=slice(None)):
    """Probabilities of one config: weighted ensemble (`weights`) or the reference model,
    optionally multiplied per class (`scale`)."""
    if "weights" in params:
        proba = sum(w * P[n][idx] for w, n in zip(params["weights"], names))
    else:
        proba = P[REF][idx]
    if "scale" in params:
        proba = proba * np.array([params["scale"].get(label, 1.0) for label in LABELS])
    return proba


def crossfit(grid):
    """Pick params on one half of val, score on the other half (both ways); then pick final params on all val."""
    oof = np.empty(len(y_val), dtype=object)
    for a, b in StratifiedKFold(2, shuffle=True, random_state=SEED).split(np.zeros(len(y_val)), y_val):
        best = max(grid, key=lambda g: macro(y_val[a], combine(P_val, g, a)))
        oof[b] = np.array(LABELS)[combine(P_val, best, b).argmax(1)]
    final = max(grid, key=lambda g: macro(y_val, combine(P_val, g)))
    return f1_score(y_val, oof, average="macro"), final

In [6]:
GRIDS = {
    "ensemble": [{"weights": w} for w in simplex(len(names), WEIGHT_STEP)],
    "scaling": [{"scale": s} for s in scale_grid(SCALE_GRID)],
    "ensemble+scaling": [{"weights": w, "scale": s}
                         for w in simplex(len(names), JOINT_WEIGHT_STEP) for s in scale_grid(JOINT_SCALE_GRID)],
}
rows, params = [{"experiment": f"reference ({REF})", "macro_f1": single[REF]}], {}
for name, grid in GRIDS.items():
    score, params[name] = crossfit(grid)
    rows.append({"experiment": name, "macro_f1": score, "grid_size": len(grid)})
results = pd.DataFrame(rows)
results["delta_vs_ref"] = results["macro_f1"] - single[REF]
results.round(4)

,experiment,macro_f1,grid_size,delta_vs_ref
0,reference (baseline_full),0.8951,NaN,0.0000
1,ensemble,0.9322,11.0,0.0371
2,scaling,0.8955,256.0,0.0004
3,ensemble+scaling,0.9347,245.0,0.0396


## Stacking

- Meta-model: `LogisticRegression` on log-probabilities of every model, one-hot `domain` and typography artifact counts ([`artifact_features`](https://github.com/mohinska/ai_hw3_AIGenTextDetection/blob/main/src/features.py))
- Why: domains behave very differently (`arxiv_papers` 84% `ai`), and `06` showed that typography drives the predictions
- Same 2-fold split of val as above; scaler and one-hot are fitted on the training half only

In [7]:
def stack_frame(df, P):
    """Meta-features: typography artifacts, log-probabilities of every model, domain."""
    X = artifact_features(df[TEXT_COL]).add_prefix("art_")
    for n in names:
        for j, label in enumerate(LABELS):
            X[f"{n}_{label}"] = np.log(np.clip(P[n][:, j], 1e-6, 1.0))
    X["domain"] = df["domain"].to_numpy()
    return X


S_val, S_test = stack_frame(val, P_val), stack_frame(test_df, P_test)
NUM_COLS = [c for c in S_val.columns if c != "domain"]


def make_stacker(class_weight):
    """Scaler + one-hot + LogReg, all fitted on whatever rows are passed to .fit()."""
    pre = ColumnTransformer([("num", StandardScaler(), NUM_COLS),
                             ("dom", OneHotEncoder(handle_unknown="ignore"), ["domain"])])
    return make_pipeline(pre, LogisticRegression(C=STACK_C, class_weight=class_weight, max_iter=2000,
                                                 random_state=SEED))


stack_test = {}
for name, cfg in STACKERS.items():
    oof = np.empty(len(y_val), dtype=object)
    for a, b in StratifiedKFold(2, shuffle=True, random_state=SEED).split(S_val, y_val):
        oof[b] = make_stacker(cfg["class_weight"]).fit(S_val.iloc[a], y_val[a]).predict(S_val.iloc[b])
    rows.append({"experiment": name, "macro_f1": f1_score(y_val, oof, average="macro"), "grid_size": None})
    meta = make_stacker(cfg["class_weight"]).fit(S_val, y_val)   # final meta-model on the whole val
    stack_test[name] = meta.predict_proba(S_test)[:, [list(meta.classes_).index(l) for l in LABELS]]

results = pd.DataFrame(rows)
results["delta_vs_ref"] = results["macro_f1"] - single[REF]
results.round(4)

,experiment,macro_f1,grid_size,delta_vs_ref
0,reference (baseline_full),0.8951,NaN,0.0000
1,ensemble,0.9322,11.0,0.0371
2,scaling,0.8955,256.0,0.0004
3,ensemble+scaling,0.9347,245.0,0.0396
4,stacking,0.9361,NaN,0.0411
5,stacking_balanced,0.9361,NaN,0.0410


In [8]:
# chosen parameters (fitted on the whole validation split)
chosen = {name: {"weights": dict(zip(names, np.round(p["weights"], 2))) if "weights" in p else None,
                 "scale": p.get("scale")}
          for name, p in params.items()}
os.makedirs(os.path.join(common.WORK_DIR, "outputs"), exist_ok=True)
with open(os.path.join(common.WORK_DIR, "outputs", "ensemble_params.json"), "w") as f:
    json.dump(chosen, f, indent=1, default=float)
pd.DataFrame(chosen).T

,weights,scale
ensemble,"{'baseline_full': 0.5, 'encoder_full_chunks': ...",None
scaling,None,"{'human': 1.6, 'fusion': 1.5}"
ensemble+scaling,"{'baseline_full': 0.5, 'encoder_full_chunks': ...","{'human': 1.25, 'fusion': 1.0}"


## Test predictions for the best experiment

In [9]:
best_row = results.iloc[1:].sort_values("macro_f1", ascending=False).iloc[0]
BEST = best_row.experiment if best_row.macro_f1 > single[REF] else "reference"   # fall back if nothing beats it
if BEST == "reference":
    test_proba = P_test[REF]
elif BEST in stack_test:
    test_proba = stack_test[BEST]
else:
    test_proba = combine(P_test, params[BEST])

submission = common.save_submission(test_df["id"], np.array(LABELS)[test_proba.argmax(1)], sample_sub, SUBMISSION_PATH)
print("best:", BEST, "| saved", SUBMISSION_PATH, submission.shape)
submission["label"].value_counts(normalize=True).round(3)

best: stacking | saved /kaggle/working/submission_ensemble.csv (25000, 2)


label
ai        0.514
human     0.260
fusion    0.226
Name: proportion, dtype: float64

## Conclusions

Subsample run (baseline 60k + encoder 40k, outputs replaced by the final run):

- **Ensemble is the biggest gain of the project:** `ensemble+scaling` 0.9004 vs 0.8725 for the best single model (+0.028), LB 0.89673
- **Encoder is essential:** without it (only TF-IDF models) the ensemble gave +0.0003; with it +0.026 -> the two models make different errors
- **Chosen params:** `baseline` 0.5 + `encoder_chunks` 0.5, `p_human` x2.0, `p_fusion` x1.5 -> both minority classes are boosted against `ai`
- **Scaling alone does not help** (-0.0003): one model cannot be fixed by moving thresholds, the gain comes from combining
- **`tfidf_class_weight` gets weight 0** in the joint search: it duplicates `baseline`

Final run (`baseline_full` + `encoder_full_chunks`, outputs above):

- **Best: `stacking`** 0.9361 on val (+0.041 vs `baseline_full`), LB 0.93481 -> val and LB agree
- **Stacking vs weighted average:** +0.0014 over `ensemble+scaling` (0.9347) -> `domain` and typography features add a little on top of the two models
- **Scaling alone still gives nothing** (+0.0004): the gain comes from combining two complementary models